In [2]:
# =========================
# 1. IMPORTS
# =========================
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Lambda
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input


In [3]:
# =========================
# 2. DATA LOADING + SPLIT
# =========================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# This automatically splits into train + validation
#note: tf.keras.preprocessing.image_dataset_from_directory(...) this automatically loads the image and converts it into RGB internally
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "cnn_dataset",
    validation_split=0.2,     # 80% train, 20% validation
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_ds.class_names
print(class_names)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "cnn_dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

Found 160574 files belonging to 2 classes.
Using 128460 files for training.
['faulty', 'good']
Found 160574 files belonging to 2 classes.
Using 32114 files for validation.


In [4]:
# =========================
# 3. PERFORMANCE OPTIMIZATION
# =========================

# speeds up training (load next batch while training current batch) THIS IS OPTIONAL JUST WILL MAKE IT FASTER
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [5]:
# =========================
# 4. MODEL BUILDING (Functional API)
# =========================

# Input layer (image shape)
inputs = Input(shape=(224, 224, 3))


# Load pretrained EfficientNet (feature extractor)
base_model = EfficientNetB0(
    include_top=False,       # remove original classifier
    weights='imagenet'       # use pretrained knowledge
)

# Freeze base model (Phase 1)
base_model.trainable = False

# Preprocessing layer (IMPORTANT: matches EfficientNet expectations)
x = Lambda(preprocess_input)(inputs)  

# Pass input through EfficientNet
x = base_model(x, training=False)  #training false so model dosnt update

# Convert 3D feature maps → 1D vector this is more efficient version of flatten() method
x = GlobalAveragePooling2D()(x)

# Regularization to prevent overfitting
x = Dropout(0.3)(x)

# Final classification layer (binary: good vs faulty)
outputs = Dense(1, activation='sigmoid')(x)

# Create full model
model = Model(inputs, outputs)


In [6]:
# =========================
# 5. COMPILE (PHASE 1)
# =========================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [7]:
# =========================
# 6. TRAIN (PHASE 1)
# =========================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)
 

Epoch 1/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 2094s 521ms/step - accuracy: 0.9690 - loss: 0.0853 - val_accuracy: 0.9872 - val_loss: 0.0396
Epoch 2/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 2674s 666ms/step - accuracy: 0.9805 - loss: 0.0533 - val_accuracy: 0.9897 - val_loss: 0.0314
Epoch 3/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 9290s 2s/step - accuracy: 0.9817 - loss: 0.0508 - val_accuracy: 0.9897 - val_loss: 0.0294
Epoch 4/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 2162s 538ms/step - accuracy: 0.9817 - loss: 0.0495 - val_accuracy: 0.9909 - val_loss: 0.0274
Epoch 5/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 1896s 472ms/step - accuracy: 0.9826 - loss: 0.0484 - val_accuracy: 0.9910 - val_loss: 0.0269


In [8]:
# =========================
# 7. FINE-TUNING (PHASE 2)
# =========================

# Unfreeze the base model
base_model.trainable = True

# OPTIONAL: freeze early layers (keep basic features)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with LOW learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train again (fine-tuning)
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 4732s 1s/step - accuracy: 0.9732 - loss: 0.0740 - val_accuracy: 0.9958 - val_loss: 0.0131
Epoch 2/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 5900s 1s/step - accuracy: 0.9924 - loss: 0.0205 - val_accuracy: 0.9973 - val_loss: 0.0080
Epoch 3/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 4008s 998ms/step - accuracy: 0.9960 - loss: 0.0112 - val_accuracy: 0.9979 - val_loss: 0.0063
Epoch 4/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 5740s 1s/step - accuracy: 0.9975 - loss: 0.0070 - val_accuracy: 0.9982 - val_loss: 0.0054
Epoch 5/5
4015/4015 ━━━━━━━━━━━━━━━━━━━━ 3562s 887ms/step - accuracy: 0.9982 - loss: 0.0051 - val_accuracy: 0.9982 - val_loss: 0.0053


In [9]:
model.save("cnn_model.keras")


In [10]:
import json

with open("class_names.json", "w") as f:
    json.dump(class_names, f)


In [16]:
#VERIFYING PREDICTIONS

import numpy as np
from tensorflow.keras.preprocessing import image

# path to your test image
img_path = "daru.jpeg"

# load image
img = image.load_img(img_path, target_size=(224, 224))

# convert to array
img_array = image.img_to_array(img)

# add batch dimension (VERY IMPORTANT)
img_array = np.expand_dims(img_array, axis=0)

# preprocess (same as training)
img_array = preprocess_input(img_array)

prediction = model.predict(img_array)
prob = prediction[0][0]

print("Raw prediction:", prob)

if prob > 0.5:
    print("Predicted: Good ✅")
else:
    print("Predicted: Faulty ❌")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Raw prediction: 0.00020269453
Predicted: Faulty ❌
